# Phase 7 — Knowledge distillation: a 4x narrower student of Model E, with and without distillation
* Teacher: the final Model E of each configuration (outputs_v3/final_v1/models/, Phase 5)
* Student: the same architecture as Model E (2x ECA + skip branch) with every width divided by 4
  (Conv1D 16 and 32 filters, skip Dense 32, Dense 16): about 90-92% fewer parameters, close to the 92-95%
  reduction of Wisanwanichthan & Thammawichai (2025)
* Distillation loss (Hinton et al., 2015; Wisanwanichthan & Thammawichai, 2025):
  L = alpha * T^2 * KL(p_teacher^T || p_student^T) + (1 - alpha) * CE(y, p_student), with T = 3 and alpha = 0.4
  (T = 3 also in Khan et al., 2025). Softened probabilities are computed as softmax(log p / T), which equals
  softmax(z / T) for the logits z.
* Baseline: the same student trained without distillation (alpha = 0, cross-entropy only), as in
  Wisanwanichthan & Thammawichai (2025), so that the effect of distillation can be measured
* Class weights w_c = N / (K * N_c) on the training split (Kamal & Mashaly, 2025), applied as sample weights;
  Adam 1e-4, batch 128, up to 100 epochs, dropout 0.3, early stopping with patience 5 on the unweighted
  cross-entropy of the complete validation split (as in Phases 4 and 5)
* Same logs and epoch checkpoints as Phase 5; re-run Cell 2 after any interruption.
Outputs: outputs_v3/kd_v1/ (results/, checkpoints/, models/, kd_v1_log.txt, summary.json)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings, student model, helpers
import os, json, time, gc, math, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

FEAT    = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
TEACHER = '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/'
OUT     = '/content/drive/MyDrive/Thesis/outputs_v3/kd_v1/'
for sub in ('results/', 'checkpoints/', 'models/'):
    os.makedirs(OUT + sub, exist_ok=True)
SEED, LR, BATCH, EPOCHS, PATIENCE = 42, 1e-4, 128, 100, 5
T_KD, ALPHA, WIDTH_DIV = 3.0, 0.4, 4
N_CLASSES = 8
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
SRC = {'cicids': 'CICIDS2017', 'iot': 'CIC-IoT2023'}
CONFIGS = ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']
RUNS = [(f'{m}_{c}', m, c) for c in CONFIGS for m in ('kd', 'nokd')]   # (name, mode, configuration)
TOTAL_RUNS = len(RUNS)
LOGFILE = OUT + 'kd_v1_log.txt'

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:                        # the full log is also kept on Drive
        f.write(line + '\n')

def hm(minutes):
    minutes = max(0.0, float(minutes))
    return f'{int(minutes // 60)} h {int(round(minutes % 60)):02d} min' if minutes >= 60 else f'{minutes:.1f} min'

def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build(variant, n_features, n_classes=N_CLASSES):
    """A: CNN+GAP | B: +ECA | C: +skip | D: +ECA +skip | E: +2xECA +skip (identical to the ablation study)"""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    if variant == 'E':
        x = eca(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if variant in ('B', 'D', 'E'):
        x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    if variant in ('C', 'D', 'E'):
        s = layers.Dense(128, activation='relu')(layers.Flatten()(inp))
        z = layers.Concatenate()([z, s])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Model_{variant}')

def counts(y):
    c = np.bincount(y, minlength=N_CLASSES)
    return {CLASS_NAMES[i]: int(n) for i, n in enumerate(c) if n}

def class_weights(y):
    """w_c = N / (K * N_c) over the K classes present in the training data (Kamal & Mashaly, 2025)."""
    c = np.bincount(y, minlength=N_CLASSES); present = np.flatnonzero(c)
    return {int(k): float(len(y) / (len(present) * c[k])) for k in present}

def result_path(name):
    return OUT + f'results/{name}.json'

def ckpt_dir(name):
    return OUT + f'checkpoints/{name}/'

def n_done():
    return sum(os.path.exists(result_path(r[0])) for r in RUNS)

def save_atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def val_metrics(model, Xv, yv):
    """Loss, accuracy, Macro F1 and per-class F1 on the complete validation split (one prediction pass)."""
    p = model.predict(Xv, batch_size=8192, verbose=0)
    loss = float(-np.mean(np.log(np.clip(p[np.arange(len(yv)), yv], 1e-7, 1.0))))   # unweighted cross-entropy
    yp = p.argmax(1); labels = np.unique(yv)
    f1c = f1_score(yv, yp, average=None, labels=labels, zero_division=0)
    return loss, float(accuracy_score(yv, yp)), float(f1c.mean()), {CLASS_NAMES[k]: round(float(v), 3) for k, v in zip(labels, f1c)}

def evaluate(model, X, y, labels=None):
    """Metrics on a test split; labels = the classes over which Macro F1 is averaged (default: those present in y)."""
    t0 = time.time()
    pred = model.predict(X, batch_size=8192, verbose=0).argmax(1)
    sec = time.time() - t0
    labels = np.unique(y) if labels is None else np.asarray(labels)
    pr, rc, f1c, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)),
                weighted_f1=float(f1_score(y, pred, average='weighted', labels=labels, zero_division=0)),
                per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), support=int(s_))
                           for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)},
                confusion_matrix=confusion_matrix(y, pred, labels=labels).tolist(),
                labels=[CLASS_NAMES[k] for k in labels], test_rows=int(len(y)),
                predicted_outside_labels=int((~np.isin(pred, labels)).sum()),
                inference_seconds=round(sec, 2), inference_ms_per_1000=round(1000 * 1000 * sec / max(1, len(y)), 3))

def log_eval(prefix, r):
    log(f"{prefix}: Macro F1 {r['macro_f1']:.4f} | accuracy {r['accuracy']:.4f} | weighted F1 {r['weighted_f1']:.4f} | "
        f"{r['test_rows']:,} test rows | inference {r['inference_ms_per_1000']:.3f} ms per 1,000 records")
    log('   per class (precision / recall / F1 / support): ' + '; '.join(
        f"{k} {v['precision']:.3f}/{v['recall']:.3f}/{v['f1']:.3f}/{v['support']:,}" for k, v in r['per_class'].items()))
    if r['predicted_outside_labels']:
        log(f"   {r['predicted_outside_labels']:,} records were predicted as a class outside the evaluated classes (counted as errors)")

# ---------- data

def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build_student(n_features, div=WIDTH_DIV, n_classes=N_CLASSES):
    """Model E (Conv 64 -> ECA -> Conv 128 -> ECA -> GAP, skip Dense 128, Dense 64) with every width divided by div."""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64 // div, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = eca(x)
    x = layers.Conv1D(128 // div, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    s = layers.Dense(128 // div, activation='relu')(layers.Flatten()(inp))
    z = layers.Concatenate()([z, s])
    z = layers.Dense(64 // div, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Student_E_div{div}')

def _kd(y_true, y_pred, alpha):
    """y_true = [label, teacher probabilities (8)]; y_pred = student probabilities."""
    lab = keras.ops.cast(y_true[:, 0], 'int32'); pt = y_true[:, 1:]
    ce = keras.losses.sparse_categorical_crossentropy(lab, y_pred)
    if alpha == 0:
        return ce
    ls = keras.ops.log(keras.ops.clip(y_pred, 1e-7, 1.0)); lt = keras.ops.log(keras.ops.clip(pt, 1e-7, 1.0))
    ps, pt_t = keras.ops.softmax(ls / T_KD, axis=-1), keras.ops.softmax(lt / T_KD, axis=-1)
    kl = keras.ops.sum(pt_t * (keras.ops.log(keras.ops.clip(pt_t, 1e-7, 1.0)) - keras.ops.log(keras.ops.clip(ps, 1e-7, 1.0))), axis=-1)
    return alpha * T_KD ** 2 * kl + (1 - alpha) * ce

@tf.keras.utils.register_keras_serializable(package='kd')
def kd_loss(y_true, y_pred):
    return _kd(y_true, y_pred, ALPHA)

@tf.keras.utils.register_keras_serializable(package='kd')
def ce_only_loss(y_true, y_pred):
    return _kd(y_true, y_pred, 0.0)

def load_config(cfg, split):
    X = np.load(FEAT + cfg + f'/X_{split}.npy').astype(np.float32, copy=False)
    y = np.load(FEAT + cfg + f'/y_{split}.npy').astype(np.int64)
    return X, y

class ProgressCheckpoint(keras.callbacks.Callback):
    """Detailed progress (every 10% of an epoch, with time left), complete-validation evaluation, early stopping
    (patience 5, best weights kept) and an atomic checkpoint after every epoch."""
    def __init__(self, tag, d, Xv, yv, steps, state):
        super().__init__()
        self.tag, self.d, self.Xv, self.yv, self.steps, self.state = tag, d, Xv, yv, steps, state
        self.marks = {max(1, int(steps * q / 10)) for q in range(1, 10)}
        self.t_run = time.time() - state['minutes'] * 60
    def on_epoch_begin(self, epoch, logs=None):
        self.t_ep, self.cur = time.time(), epoch + 1
        log(f"{self.tag} | epoch {self.cur} started ({self.steps:,} steps of {BATCH})")
    def on_train_batch_end(self, batch, logs=None):
        if batch + 1 in self.marks:
            el = (time.time() - self.t_ep) / 60; frac = (batch + 1) / self.steps
            log(f"{self.tag} | epoch {self.cur} | step {batch + 1:,}/{self.steps:,} ({100 * frac:.0f}%) | "
                f"running loss {logs['loss']:.4f} | {hm(el)} elapsed, about {hm(el / frac - el)} left in this epoch")
    def on_epoch_end(self, epoch, logs=None):
        st, d = self.state, self.d
        t_train = (time.time() - self.t_ep) / 60
        vloss, vacc, vf1, vpc = val_metrics(self.model, self.Xv, self.yv)
        improved = vloss < st['best']
        if improved:
            self.model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
            st.update(best=vloss, best_ep=epoch + 1, wait=0)
        else:
            st['wait'] += 1
        st.update(epoch=epoch + 1, minutes=(time.time() - self.t_run) / 60)
        st.setdefault('history', []).append(dict(epoch=epoch + 1, loss=round(float(logs['loss']), 5), val_loss=round(vloss, 5),
                                                 val_acc=round(vacc, 5), val_macro_f1=round(vf1, 5)))
        self.model.save(d + 'last.tmp.keras'); os.replace(d + 'last.tmp.keras', d + 'last.keras')
        save_atomic_json(st, d + 'state.json')
        ep_min = st['minutes'] / st['epoch']
        log(f"{self.tag} | epoch {epoch + 1} done | train {t_train:.1f} min + validation {(time.time() - self.t_ep) / 60 - t_train:.1f} min | "
            f"loss {logs['loss']:.4f} | val_loss {vloss:.4f}{' (best)' if improved else ''} | val_acc {vacc:.4f} | val_macroF1 {vf1:.4f}")
        log(f"{self.tag} |   val F1 per class: {vpc}")
        log(f"{self.tag} |   best epoch {st['best_ep']} | no improvement {st['wait']}/{PATIENCE} | run time {hm(st['minutes'])} | "
            f"at most {PATIENCE - st['wait']} more epochs without improvement (about {hm((PATIENCE - st['wait']) * ep_min)}) | checkpoint saved")
        if st['wait'] >= PATIENCE:
            log(f"{self.tag} | early stopping: no improvement for {PATIENCE} epochs; best epoch {st['best_ep']} will be restored")
            self.model.stop_training = True


log('=' * 100)
log(f'Knowledge distillation v1 | TensorFlow {tf.__version__} | Python {platform.python_version()} | GPU: '
    f'{[g.name for g in tf.config.list_physical_devices("GPU")] or "none"}')
log(f'settings: T {T_KD}, alpha {ALPHA}, student width / {WIDTH_DIV}, lr {LR}, batch {BATCH}, max epochs {EPOCHS}, '
    f'patience {PATIENCE}, class weights N/(K*N_c) | {n_done()} of {TOTAL_RUNS} runs already finished')

In [ ]:
# %% Cell 2 — train the students (resumable: re-run this cell after any interruption)
T0 = time.time(); cache = {}
for i, (name, mode, cfg) in enumerate(RUNS, 1):
    if os.path.exists(result_path(name)):
        log(f'run {i} of {TOTAL_RUNS} ({name}): already finished, skipping'); continue
    t_load = time.time()
    if cache.get('cfg') != cfg:
        cache.clear(); gc.collect()
        Xtr, ytr = load_config(cfg, 'train'); Xv, yv = load_config(cfg, 'val'); Xte, yte = load_config(cfg, 'test')
        teacher = keras.models.load_model(TEACHER + f'models/final_{cfg}.keras')
        pt = teacher.predict(Xtr[..., None], batch_size=8192, verbose=0).astype(np.float32)
        t_val = evaluate(teacher, Xv[..., None], yv)['macro_f1']
        cache = dict(cfg=cfg, Xtr=Xtr[..., None], ytr=ytr, Xv=Xv[..., None], yv=yv, Xte=Xte[..., None], yte=yte,
                     ypack=np.concatenate([ytr[:, None].astype(np.float32), pt], 1),
                     teacher_params=int(sum(np.prod(v.shape) for v in teacher.trainable_weights)))
        log(f'{cfg}: teacher (Model E) loaded | {cache["teacher_params"]:,} parameters | teacher validation Macro F1 {t_val:.4f} | '
            f'teacher probabilities computed for {len(ytr):,} training rows | loaded in {time.time() - t_load:.0f}s')
        del teacher, pt; gc.collect()
    c = cache
    cw = class_weights(c['ytr'])
    wvec = np.zeros(N_CLASSES, np.float32)
    for k, v in cw.items():
        wvec[k] = v
    sample_w = wvec[c['ytr']]
    steps = int(math.ceil(len(c['ytr']) / BATCH))
    loss_fn = kd_loss if mode == 'kd' else ce_only_loss
    log(f'--- run {i} of {TOTAL_RUNS}: {name} ({"with distillation, T = %g, alpha = %g" % (T_KD, ALPHA) if mode == "kd" else "without distillation (cross-entropy only)"}) ---')
    log(f'{name}: training {len(c["ytr"]):,} rows | validation {len(c["yv"]):,} | test {len(c["yte"]):,} | class weights ' +
        str({CLASS_NAMES[k]: round(v, 4) for k, v in cw.items()}))
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    d = ckpt_dir(name); os.makedirs(d, exist_ok=True)
    if os.path.exists(d + 'state.json') and os.path.exists(d + 'last.keras'):
        state = json.load(open(d + 'state.json'))
        model = keras.models.load_model(d + 'last.keras')
        ep_model = int(keras.ops.convert_to_numpy(model.optimizer.iterations)) // steps
        log(f"{name} --- RESUMING after epoch {state['epoch']} (best epoch {state['best_ep']}, "
            f"no improvement {state['wait']}/{PATIENCE}, {hm(state['minutes'])} already trained) ---")
        if ep_model == state['epoch'] + 1:
            log(f'{name} | the saved model is one epoch ahead of the saved state; evaluating it to bring the state up to date')
            vloss = val_metrics(model, c['Xv'], c['yv'])[0]
            if vloss < state['best']:
                model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
                state.update(best=vloss, best_ep=ep_model, wait=0)
            else:
                state['wait'] += 1
            state['epoch'] = ep_model; save_atomic_json(state, d + 'state.json')
        elif ep_model != state['epoch']:
            log(f"{name} | note: optimizer step count corresponds to epoch {ep_model}, state says {state['epoch']}; continuing from the state")
    else:
        state = dict(epoch=0, best=float('inf'), best_ep=0, wait=0, minutes=0.0, history=[])
        model = build_student(c['Xtr'].shape[1])
        model.compile(optimizer=keras.optimizers.Adam(LR), loss=loss_fn)
    params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
    log(f'{name} | student {params:,} trainable parameters ({100 * (1 - params / c["teacher_params"]):.1f}% fewer than the teacher) | {steps:,} steps per epoch')
    if not (state['wait'] >= PATIENCE or state['epoch'] >= EPOCHS):
        model.fit(c['Xtr'], c['ypack'], sample_weight=sample_w, batch_size=BATCH, epochs=EPOCHS, initial_epoch=state['epoch'],
                  verbose=0, shuffle=True, callbacks=[ProgressCheckpoint(name, d, c['Xv'], c['yv'], steps, state)])
    model.load_weights(d + 'best.weights.h5')
    model.save(OUT + f'models/{name}.tmp.keras'); os.replace(OUT + f'models/{name}.tmp.keras', OUT + f'models/{name}.keras')
    size_kb = os.path.getsize(OUT + f'models/{name}.keras') / 1024
    res = dict(run=name, mode=mode, config=cfg, T=T_KD if mode == 'kd' else None, alpha=ALPHA if mode == 'kd' else 0.0,
               epochs=state['epoch'], best_epoch=state['best_ep'], train_minutes=round(state['minutes'], 2), params=params,
               teacher_params=c['teacher_params'], reduction_percent=round(100 * (1 - params / c['teacher_params']), 2),
               model_file_kb=round(size_kb, 1), history=state.get('history', []), tests={})
    r = evaluate(model, c['Xte'], c['yte']); res['tests']['own'] = r
    log_eval(f'{name} RESULT [test: {cfg}]', r)
    if cfg == 'config1_merged':
        s = np.load(FEAT + cfg + '/source_test.npy')
        for src in (SRC['cicids'], SRC['iot']):
            m = s == src
            r = evaluate(model, c['Xte'][m], c['yte'][m]); res['tests'][f'merged_test_{src}'] = r
            log_eval(f'{name} RESULT [merged test split, {src} rows only]', r)
    save_atomic_json(res, result_path(name))
    log(f"{name}: {res['epochs']} epochs (best {res['best_epoch']}) | trained in {hm(res['train_minutes'])} | model file {size_kb:.0f} KB")
    log(f'   finished {n_done()} of {TOTAL_RUNS} runs | this session running for {hm((time.time() - T0) / 60)}')
    del model; gc.collect()
log(f'knowledge distillation v1: {n_done()} of {TOTAL_RUNS} runs finished')

In [ ]:
# %% Cell 3 — summary: teacher vs student with and without distillation
rows = []
for cfg in CONFIGS:
    t = json.load(open(TEACHER + f'results/final_{cfg}.json'))
    rows.append(dict(configuration=cfg, model='teacher (Model E)', macro_f1=t['tests']['own']['macro_f1'], accuracy=t['tests']['own']['accuracy'],
                     weighted_f1=t['tests']['own']['weighted_f1'], params=t['params'], model_kb=t['model_file_kb'],
                     ms_per_1000=t['tests']['own']['inference_ms_per_1000']))
    for mode, label in (('kd', 'student with KD'), ('nokd', 'student without KD')):
        p = result_path(f'{mode}_{cfg}')
        if os.path.exists(p):
            r = json.load(open(p))
            rows.append(dict(configuration=cfg, model=label, macro_f1=r['tests']['own']['macro_f1'], accuracy=r['tests']['own']['accuracy'],
                             weighted_f1=r['tests']['own']['weighted_f1'], params=r['params'], model_kb=r['model_file_kb'],
                             ms_per_1000=r['tests']['own']['inference_ms_per_1000']))
tab = pd.DataFrame(rows)
print(f'{n_done()} of {TOTAL_RUNS} student runs finished\n')
print(tab.round(4).to_string(index=False))
save_atomic_json(dict(table=rows), OUT + 'summary.json')